In [9]:
import json

import re
import pickle
import json
import numpy as np
import pandas as pd

import torch
from transformers import AutoTokenizer
from copy import deepcopy

import sys
sys.path.append("/root/workspace/code/reason/")
from evol.fitness import extract_answer
from evol.fitness import cosine_scaled_reward, accuracy_reward, format_reward, cosine_lang_reward
# from recipes.test.evol_data_analys import calculate_fitness

In [14]:
math_message = (
    "You are an expert in math. "
    "Below is a math question. Write a response that appropriately answers the question. "
    "The final answer should be formatted as $\\boxed{{}}$. Let's think step by step."
)
tokenizer = AutoTokenizer.from_pretrained("/media/user/pretrained/Qwen2.5-7B-Instruct")

In [10]:
data = []
with open("/media/user/data/ga/math7500/math7500.json") as file:
    for line in file:
        data.append(json.loads(line))

In [13]:
data[0]

{'problem_id': 5850,
 'question': 'Simplify $90r - 44r$.',
 'gt_answer': '46r',
 'gt_solution': 'The difference between $44r$ and $90r$ is $90r-44r=\\boxed{46r}$.',
 'solution_id': 0,
 'solution': 'First, I need to simplify the expression \\(90r - 44r\\).\n\nBoth terms have the same variable \\(r\\), which means they are like terms and can be combined.\n\nI will subtract the coefficients: \\(90 - 44 = 46\\).\n\nTherefore, the simplified expression is \\(46r\\).\n</think>\n\nTo simplify the expression \\(90r - 44r\\), follow these steps:\n\n1. **Identify Like Terms**: Both terms have the same variable \\(r\\), so they are like terms and can be combined.\n\n2. **Subtract the Coefficients**:\n   \\[\n   90r - 44r = (90 - 44)r\n   \\]\n\n3. **Calculate the Coefficient**:\n   \\[\n   90 - 44 = 46\n   \\]\n\n4. **Write the Simplified Expression**:\n   \\[\n   46r\n   \\]\n\n**Final Answer:**\n\\[\n\\boxed{46r}\n\\]',
 'is_correct': True}

### 测试

In [7]:
eval_data = []
with open("/media/user/data/ga/math7500/test.jsonl") as file:
    for line in file:
        eval_data.append(json.loads(line))

eval_dps = []
for dp in eval_data:
    eval_dp = {"prompt": dp["problem"], "response": dp["solution"]}
    eval_dps.append(eval_dp)
eval_dps[0]

{'prompt': 'What is $10.0000198\\cdot 5.9999985401\\cdot 6.9999852$ to the nearest whole number?',
 'response': "Notice that $10.00001988$ is very close to $10$, $5.9999985401$ is very close to $6$ and $6.9999852$ is very close to $7$. Because the given numbers are all so close to integers, we're unlikely to go wrong by rounding before multiplying. We get $$10\\cdot6\\cdot7=\\boxed{420}.$$If we multiplied the given numbers with a calculator we would get $$6.9999852\\cdot5.9999985401\\cdot10.00001988=419.999844...$$which would still round to $420$."}

### short math

In [15]:
cnt = 0
train_dps = []
train_len = []
resp_name = "gt_solution"
for idx, dp in enumerate(data):
    answer = extract_answer(dp[resp_name])
    if not answer:
        continue
    train_dp = {"prompt": dp["question"], "response": dp[resp_name], "idx": idx}
    example = {
            "prompt": [{"role": "system", "content": math_message}, {"role": "user", "content": train_dp['prompt']}],
            "response": [{"role": "assistant", "content": train_dp['response']}, ],
        }
    train_len.append(len(tokenizer.apply_chat_template(example["prompt"] + example["response"], tokenize=True, add_generation_prompt=True)))
    train_dps.append(train_dp)

In [18]:
lower_q=np.quantile(train_len, 0.95, interpolation='lower')#下四分位数
lower_q

839

In [19]:
torch.save((train_dps, eval_dps), f"/media/user/data/ga/math7500/math7500_short_cot.pt")

In [20]:
len(train_dps)

7450

### long math

In [21]:
cnt = 0
train_dps = []
train_len = []
resp_name = "solution"
for idx, dp in enumerate(data):
    answer = extract_answer(dp[resp_name])
    if not answer:
        continue
    train_dp = {"prompt": dp["question"], "response": dp[resp_name], "idx": idx}
    example = {
            "prompt": [{"role": "system", "content": math_message}, {"role": "user", "content": train_dp['prompt']}],
            "response": [{"role": "assistant", "content": train_dp['response']}, ],
        }
    train_len.append(len(tokenizer.apply_chat_template(example["prompt"] + example["response"], tokenize=True, add_generation_prompt=True)))
    train_dps.append(train_dp)

In [29]:
lower_q=np.quantile(train_len, 0.90, interpolation='lower')#下四分位数
lower_q

7995

In [24]:
torch.save((train_dps, eval_dps), f"/media/user/data/ga/math7500/math7500_long_cot.pt")

In [45]:
order = "-"
reuse_idx = 14
train_data = [i for i in range(100)]
batch_size = 7

if order == "-":
    idxs = list(reversed(list(range(len(train_data)))))
    reuse_idx = len(train_data) - reuse_idx
    start_idx, end_idx = len(idxs)-1, -1
else:
    idxs = list(range(len(train_data)))
    start_idx, end_idx = 0, len(idxs)

print(idxs)

for idx in range(start_idx, end_idx, -batch_size):
    # print(idxs[idx])
    if order == "-" and idxs[idx] > reuse_idx:
        continue
    elif order == "+" and idxs[idx] < reuse_idx:
        continue

    print(idx)

    bsz = batch_size if idx+batch_size < len(idxs) else len(idxs)-len(train_data)
    # logger.info(f"{idx}, {batch_size}, {len(dps)}, idxs: {len(idxs)}, {idx + batch_size}")
    # logger.info(f"Processing problem {idx}+{batch_size}")
    ids = idxs[idx:idxs[idx + batch_size]] if batch_size == bsz else idxs[idx:]

    print(ids)

[99, 98, 97, 96, 95, 94, 93, 92, 91, 90, 89, 88, 87, 86, 85, 84, 83, 82, 81, 80, 79, 78, 77, 76, 75, 74, 73, 72, 71, 70, 69, 68, 67, 66, 65, 64, 63, 62, 61, 60, 59, 58, 57, 56, 55, 54, 53, 52, 51, 50, 49, 48, 47, 46, 45, 44, 43, 42, 41, 40, 39, 38, 37, 36, 35, 34, 33, 32, 31, 30, 29, 28, 27, 26, 25, 24, 23, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 0]
99
[0]
92
[]
85
[]
78
[]
71
[]
64
[]
57
[]
50
[]
43
[56, 55, 54, 53, 52, 51]
36
[63, 62, 61, 60, 59, 58, 57, 56, 55, 54, 53, 52, 51, 50, 49, 48, 47, 46, 45, 44]
29
[70, 69, 68, 67, 66, 65, 64, 63, 62, 61, 60, 59, 58, 57, 56, 55, 54, 53, 52, 51, 50, 49, 48, 47, 46, 45, 44, 43, 42, 41, 40, 39, 38, 37]
22
[77, 76, 75, 74, 73, 72, 71, 70, 69, 68, 67, 66, 65, 64, 63, 62, 61, 60, 59, 58, 57, 56, 55, 54, 53, 52, 51, 50, 49, 48, 47, 46, 45, 44, 43, 42, 41, 40, 39, 38, 37, 36, 35, 34, 33, 32, 31, 30]
15
[84, 83, 82, 81, 80, 79, 78, 77, 76, 75, 74, 73, 72, 71, 70, 69, 68, 67, 66, 65, 64, 63, 62, 61, 60, 59, 58, 

In [49]:
order = "-"
reuse_idx = 0
train_data = [i for i in range(100)]
batch_size = 7

if order == "-":
    idxs = list(reversed(range(len(train_data))))  # [99, 98, ..., 0]
    # 跳过末尾 len(train_data) - reuse_idx 个样本（即 reuse_idx 是“正向剩余可用个数”）
    min_valid_idx = len(train_data) - reuse_idx    # e.g., 100 - 14 = 86
    start_idx, end_idx, step = 0, len(idxs), batch_size
else:
    idxs = list(range(len(train_data)))            # [0, 1, ..., 99]
    max_valid_idx = reuse_idx                      # 小于 reuse_idx 才跳过
    start_idx, end_idx, step = 0, len(idxs), batch_size

for i in range(start_idx, end_idx, step):
    batch = idxs[i:i+batch_size]

    if order == "-" and batch[0] > min_valid_idx:
        # 当前 batch 完全在已访问区域，跳过
        continue
    elif order == "+" and batch[-1] < reuse_idx:
        # 当前 batch 完全在已访问区域，跳过
        continue

    # 如果部分 batch 超过 reuse_idx，可以截断
    if order == "-":
        batch = [x for x in batch if x <= min_valid_idx]
    else:
        batch = [x for x in batch if x >= reuse_idx]

    print("Batch indices:", batch)

Batch indices: [99, 98, 97, 96, 95, 94, 93]
Batch indices: [92, 91, 90, 89, 88, 87, 86]
Batch indices: [85, 84, 83, 82, 81, 80, 79]
Batch indices: [78, 77, 76, 75, 74, 73, 72]
Batch indices: [71, 70, 69, 68, 67, 66, 65]
Batch indices: [64, 63, 62, 61, 60, 59, 58]
Batch indices: [57, 56, 55, 54, 53, 52, 51]
Batch indices: [50, 49, 48, 47, 46, 45, 44]
Batch indices: [43, 42, 41, 40, 39, 38, 37]
Batch indices: [36, 35, 34, 33, 32, 31, 30]
Batch indices: [29, 28, 27, 26, 25, 24, 23]
Batch indices: [22, 21, 20, 19, 18, 17, 16]
Batch indices: [15, 14, 13, 12, 11, 10, 9]
Batch indices: [8, 7, 6, 5, 4, 3, 2]
Batch indices: [1, 0]
